In [60]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn


In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report, precision_score, recall_score, f1_score, roc_auc_score
import pickle

In [62]:
df = pd.read_csv(f'{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [63]:
# Convert 'TotalCharges' to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')



In [64]:
#  Verify that there are no more NaN values in 'TotalCharges'

df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

In [65]:
#drop the customer ID
df.drop(columns=['customerID'], inplace=True)


In [66]:

df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0}).astype(int)

/tmp/ipykernel_9146/3657643670.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0}).astype(int)


In [67]:
X=df.drop(columns=['Churn'])
y=df['Churn']

In [68]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [69]:
cat_col=X.select_dtypes(include=['object']).columns.tolist()
num_col=X.select_dtypes(include=['int64','float64']).columns.tolist()


In [70]:

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_col),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_col),
    ]
)

In [71]:
rfc = RandomForestClassifier(
    n_estimators=439,
    max_depth=10,
    min_samples_split=3,
    min_samples_leaf=5,
    max_features="log2",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

In [72]:
#Pipeline
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42, k_neighbors=5)),
    ("rfc", rfc),  # your tuned model
])

In [73]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('smote', SMOTE(random_state=42)),
                ('rfc',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=10, max_features='log2',
                                        min_samples_leaf=5, min_samples_split=3,
                                        n_estimators=439, n_jobs=-1,
                                        random_state=42))])

In [74]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
# y_pred

In [75]:
# Create a single customer's data for prediction
new_customer_data = {
    'gender': ['Male'],
    'SeniorCitizen': [0],
    'Partner': ['Yes'],
    'Dependents': ['No'],
    'tenure': [30],
    'PhoneService': ['Yes'],
    'MultipleLines': ['Yes'],
    'InternetService': ['Fiber optic'],
    'OnlineSecurity': ['No'],
    'OnlineBackup': ['Yes'],
    'DeviceProtection': ['No'],
    'TechSupport': ['No'],
    'StreamingTV': ['Yes'],
    'StreamingMovies': ['Yes'],
    'Contract': ['Month-to-month'],
    'PaperlessBilling': ['Yes'],
    'PaymentMethod': ['Electronic check'],
    'MonthlyCharges': [85.5],
    'TotalCharges': [2500.0]
}

new_df = pd.DataFrame(new_customer_data)
print("New customer data created:")
display(new_df)

# Predict using the properly trained pipeline
new_predictions = pipeline.predict(new_df)
new_probabilities = pipeline.predict_proba(new_df)[:, 1]

print(f"Prediction (1=Churn, 0=Stay): {new_predictions[0]}")
print(f"Churn Probability: {new_probabilities[0]:.2%}")

New customer data created:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,Yes,No,30,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,85.5,2500.0


Prediction (1=Churn, 0=Stay): 1
Churn Probability: 78.32%



**Model** Prediction: 1 (Churn)

**Churn** Probability: 78.32%

১. সিদ্ধান্ত (Prediction): কাস্টমারটি কোম্পানি ছেড়ে চলে যাবেন (Churn = 1)

২. সম্ভাবনা (Probability): টেস্ট কাস্টমারের ক্ষেত্রে মডেলটি ৭৮.৩২% নিশ্চিত যে তিনি চলে যাবেন

In [76]:
import pickle
from google.colab import files

# ১. পাইপলাইনটিকে 'churn_model.pkl' নামে সেভ করা
with open('churn_model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

print("মডেলটি সফলভাবে সেভ হয়েছে!")

# ২. ফাইলটি সরাসরি আপনার কম্পিউটারে ডাউনলোড করা
files.download('churn_model.pkl')

মডেলটি সফলভাবে সেভ হয়েছে!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>